In [28]:
#import des librairies 
import os
import getpass
import pandas as pd
import faiss
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI
from pathlib import Path
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_classic.chains import RetrievalQA
from dotenv import load_dotenv

In [16]:
# ===============================================
# Chargement, Indexation FAISS & Maintien des ID
# ===============================================


# Charge le fichier CSV contenant les événements
fichier = "./data/evenement.csv"
# Verification si le fichier existe
if not Path(fichier).is_file() :
    print (f"Le fichier '{fichier} est introuvable . Arrêt du script")
    sys.exit() # Arret du programme
df = pd.read_csv(fichier)


print("--- Chargement et Indexation FAISS ---")

# Converti chaque ligne du DataFrame en Document LangChain
# On stocke l'ID et les détails dans les métadonnées pour garder la liaison directe
documents = []
for _, row in df.iterrows():
    # Le texte à vectoriser (colonne demandée)
    texte_vecteur = str(row['texte_a_vectoriser'])
    
    # Les métadonnées pour retrouver les détails complets dans le CSV plus tard
    metadata = {
        "id": row['id'],
        "titre": row.get('titre', ''),
        "lieu": row.get('lieu', ''),
        "ville": row.get('ville', ''),
        "url": row.get('url', ''),
        "date": row.get('date', '')
    }
    
    documents.append(Document(page_content=texte_vecteur, metadata=metadata))

print(f"{len(documents)} documents chargés depuis 'evenement.csv'.")


--- Chargement et Indexation FAISS ---
281 documents chargés depuis 'evenement.csv'.


In [17]:
#  Initialise le modèle d'embedding (multilingue)
embeddings_model = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
#Defini les dimensions et configure le HNSW index
dimension = 384  # Dimension des vecteurs pour le modèle MiniLM
index = faiss.IndexHNSWFlat(dimension, 32)
index.hnsw.efConstruction = 64
index.hnsw.efSearch = 32

In [19]:
# Extrait le texte and metadatas pour le document
texts = [doc.page_content for doc in documents]
metadatas = [doc.metadata for doc in documents]

In [20]:
# 3. Embed le texte
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = embeddings_model.embed_documents(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
# 4. Create the docstore and index mapping required by LangChain
docstore = InMemoryDocstore({
    str(i): doc for i, doc in enumerate(documents)
})
index_to_docstore_id = {i: str(i) for i in range(len(documents))}

In [22]:
# 5. Instantiate FAISS directly using your custom index
db_faiss = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id
)

In [23]:
# 6. Add the embeddings to the vector store index
db_faiss.add_embeddings(
    text_embeddings=list(zip(texts, embeddings)),
    metadatas=metadatas
)

['8e327b55-ccbf-428a-86c4-c72588b16ccf',
 '326d14f2-2b35-41f8-9150-acbb9623d026',
 'e80c9dc0-a337-4dc4-acb3-5c95563c5e3f',
 '1a9a3499-3d4c-4692-b023-fe0b9b2ed0b1',
 'd3a8eafb-9213-4bfd-b12d-417cdc2193ba',
 '4c9510b8-3b4d-4989-ae69-be4356fa0b83',
 '280ac37c-55b5-45f9-9598-301fe5d2b14c',
 'a2ba369e-caa0-49d8-9d50-785bb39ca28e',
 '61c6d749-fa75-4fc6-8db7-35b586338ec8',
 '660b8a91-bc91-4aef-b6e6-f3e2da48d109',
 '0ec5283e-54b7-445c-8af0-2bb1eadd8cd5',
 '2c1dda0a-7a24-4bd9-8537-ae3c17de9731',
 'a3e27e84-b2a5-4def-ad89-3329242a2b62',
 'a249a7d9-884b-46b7-bd18-42508ba02012',
 'fbdf9071-fc91-42aa-a614-d6d40e78f173',
 '50edaa48-d28a-415f-b419-72a4847ef0ac',
 '8e8bfa1d-d908-4140-83e1-91ff4eec10b7',
 'f5b00058-06da-4df4-b0eb-04649fda6e2a',
 '3dfc4fe6-5997-4750-a1c2-215150b8b349',
 '6e2ddddd-a521-42f4-86a7-a8d36ce05a3a',
 '6f6e85f2-c410-4bdd-b5b5-cfd1f254c00e',
 'c6ce9fef-1693-4dce-8114-f8a02b9ff63f',
 '56018e6d-6d1b-4c77-b567-d8d8d997f6a0',
 'a53d908e-5925-4126-93a9-8640b891e811',
 '04ebc1ff-2a25-

In [24]:
# Sauvegarde l'index localement pour le réutiliser si besoin
db_faiss.save_local("./data/index_openagenda_hnsw")
print("Index FAISS optimisé (HNSW) créé et sauvegardé avec succès !\n")

Index FAISS optimisé (HNSW) créé et sauvegardé avec succès !



# ==========================================
#  Chatbot  avec l'API Mistral
# ==========================================

In [29]:
print("---  Configuration du Chatbot Mistral ---")

load_dotenv()

api_key = os.getenv("MISTRAL_API_KEY")
if not api_key:
    raise ValueError(
        "La clé API  n'est pas définie dans l'environnement."
    )

---  Configuration du Chatbot Mistral ---


In [43]:

#  Initialiser le modèle de langage Mistral
llm = ChatMistralAI(
    model="mistral-tiny",  # Modèle rapide et performant pour le RAG
    temperature=0.3                # Température basse pour privilégier la précision factuelle
)

# 3. Transformer la base FAISS en récupérateur (retriever)
# On demande par exemple de récupérer les 3 événements les plus pertinents
retriever = db_faiss.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 4. Créer la chaîne de questions/réponses (RetrievalQA)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("Chatbot Mistral prêt à formuler des recommandations personnalisées !\n")

Chatbot Mistral prêt à formuler des recommandations personnalisées !



In [44]:
# ==========================================
# EXEMPLE D'UTILISATION DU CHATBOT
# ==========================================

# Question utilisateur en langage naturel
question_utilisateur = "Je cherche une exposition artistique ou culturelle originale à faire à Paris dans les prochains jours. Que me conseilles-tu ?"

print(f"Question posée : « {question_utilisateur} »\n")

Question posée : « Je cherche une exposition artistique ou culturelle originale à faire à Paris dans les prochains jours. Que me conseilles-tu ? »



In [45]:
# Exécution de la chaîne RAG
reponse = qa_chain.invoke({"query": question_utilisateur})

In [46]:



print("=== Réponse personnalisée de Mistral ===")
print(reponse["result"])

print("\n=== Événements sources utilisés (avec correspondance ID du CSV) ===")
for i, doc in enumerate(reponse["source_documents"]):
    # Grâce aux métadonnées, on fait le lien direct avec l'ID du fichier CSV d'origine
    print(f"{i+1}. [ID CSV : {doc.metadata['id']}] {doc.metadata['titre']}")
    print(f"   Lieu : {doc.metadata['lieu']} ({doc.metadata['ville']})")
    print(f"   Lien : {doc.metadata['url']}\n")

=== Réponse personnalisée de Mistral ===
D'après les informations disponibles, voici deux événements culturels originaux à Paris dans les prochains jours :

1. **"Explore le Monde à travers Paris"** :
   - **Description** : Un parcours inédit pour découvrir le monde à travers Paris, en partenariat avec le pass Culture.
   - **Lieu** : Institut suédois (11 rue Payenne).
   - **Période** : **Samedi 3 octobre à 15h00** (proche de la date actuelle, donc idéal si vous cherchez une activité pour ces prochains jours).

2. **"BISFF in Paris - Conférence : L’IA et le court-métrage coréen"** :
   - **Description** : Une conférence intéressante qui explore l'intersection entre l'innovation technologique coréenne et l'art cinématographique français, avec une réflexion sur l'impact de l'IA dans le cinéma.
   - **Lieu** : Centre Culturel Coréen (20 Rue La Boétie, 75008 Paris).
   - **Période** : **Lundi 12 octobre à 19h00** (un peu plus loin dans le temps, mais une option très originale et intellect

In [48]:
print("=" * 60)
print(" 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL ")
print("=" * 60)
print(reponse["result"].strip())
print("\n" + "-" * 60)

print(" 📚  ÉVÉNEMENTS SOURCES UTILISÉS (RAG) ")
print("-" * 60)

for i, doc in enumerate(reponse["source_documents"], start=1):
    # Affichage propre avec des encarts et des emojis
    print(f"[{i}] 📌 {doc.metadata.get('titre', 'Titre non disponible')}")
    print(f"    📍 Lieu   : {doc.metadata.get('lieu', 'Lieu non spécifié')} ({doc.metadata.get('ville', 'Paris')})")
    
    url = doc.metadata.get('url')
    if url:
        print(f"    🔗 Lien   : {url}")
    else:
        print("    🔗 Lien   : Non disponible")
        
    print()  # Ligne vide pour aérer entre chaque source

print("=" * 60)

 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL 
D'après les informations disponibles, voici deux événements culturels originaux à Paris dans les prochains jours :

1. **"Explore le Monde à travers Paris"** :
   - **Description** : Un parcours inédit pour découvrir le monde à travers Paris, en partenariat avec le pass Culture.
   - **Lieu** : Institut suédois (11 rue Payenne).
   - **Période** : **Samedi 3 octobre à 15h00** (proche de la date actuelle, donc idéal si vous cherchez une activité pour ces prochains jours).

2. **"BISFF in Paris - Conférence : L’IA et le court-métrage coréen"** :
   - **Description** : Une conférence intéressante qui explore l'intersection entre l'innovation technologique coréenne et l'art cinématographique français, avec une réflexion sur l'impact de l'IA dans le cinéma.
   - **Lieu** : Centre Culturel Coréen (20 Rue La Boétie, 75008 Paris).
   - **Période** : **Lundi 12 octobre à 19h00** (un peu plus loin dans le temps, mais une option très originale et intellectuel

In [50]:
print("=" * 60)
print(" 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL ")
print("=" * 60)
print(reponse["result"].strip())
print("\n" + "-" * 60)

print(" 📚  ÉVÉNEMENTS SOURCES UTILISÉS (RAG) ")
print("-" * 60)

for i, doc in enumerate(reponse["source_documents"], start=1):
    # Fonction locale pour nettoyer les valeurs "nan" ou vides
    def clean_meta(key, default="Non spécifié"):
        val = doc.metadata.get(key)
        # Si la valeur est None, vide, ou la chaîne "nan", on renvoie le défaut
        if val is None or str(val).strip().lower() in ["", "nan", "none"]:
            return default
        return str(val).strip()

    titre = clean_meta('titre', 'Titre non disponible')
    id_csv = clean_meta('id', 'N/A')
    lieu = clean_meta('lieu', 'Lieu non spécifié')
    ville = clean_meta('ville', '')
    url = clean_meta('url', '')

    # Affichage propre
    print(f"[{i}] 📌 {titre}")

    
    # Gérer l'affichage du lieu et de la ville proprement
    if ville and ville != "Non spécifié":
        print(f"    📍 Lieu   : {lieu} ({ville})")
    else:
        print(f"    📍 Lieu   : {lieu}")
        
    if url and url != "Non spécifié":
        print(f"    🔗 Lien   : {url}")
    else:
        print("    🔗 Lien   : Non disponible")
        
    print()  # Ligne vide pour aérer

print("=" * 60)

 🤖  RÉPONSE PERSONNALISÉE DE MISTRAL 
D'après les informations disponibles, voici deux événements culturels originaux à Paris dans les prochains jours :

1. **"Explore le Monde à travers Paris"** :
   - **Description** : Un parcours inédit pour découvrir le monde à travers Paris, en partenariat avec le pass Culture.
   - **Lieu** : Institut suédois (11 rue Payenne).
   - **Période** : **Samedi 3 octobre à 15h00** (proche de la date actuelle, donc idéal si vous cherchez une activité pour ces prochains jours).

2. **"BISFF in Paris - Conférence : L’IA et le court-métrage coréen"** :
   - **Description** : Une conférence intéressante qui explore l'intersection entre l'innovation technologique coréenne et l'art cinématographique français, avec une réflexion sur l'impact de l'IA dans le cinéma.
   - **Lieu** : Centre Culturel Coréen (20 Rue La Boétie, 75008 Paris).
   - **Période** : **Lundi 12 octobre à 19h00** (un peu plus loin dans le temps, mais une option très originale et intellectuel